<a href="https://colab.research.google.com/github/ghoshmoumita04/FineTuningSLM-PEFT-QLORA/blob/main/SLM_for_fineTuning_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))


CUDA available: True
GPU count: 1
GPU name: Tesla T4


In [9]:
!pip uninstall -y torch torchvision torchaudio
!pip cache purge




Found existing installation: torch 2.5.1+cu121
Uninstalling torch-2.5.1+cu121:
  Successfully uninstalled torch-2.5.1+cu121
Found existing installation: torchvision 0.20.1+cu121
Uninstalling torchvision-0.20.1+cu121:
  Successfully uninstalled torchvision-0.20.1+cu121
Found existing installation: torchaudio 2.5.1+cu121
Uninstalling torchaudio-2.5.1+cu121:
  Successfully uninstalled torchaudio-2.5.1+cu121
Files removed: 64


In [10]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121


Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 MB 554.0 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 108.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 115.4 MB/s eta 0:00:00


In [1]:
!pip install -U \
  transformers \
  accelerate \
  peft \
  bitsandbytes \
  datasets \
  sentencepiece


In [2]:
import torch, bitsandbytes as bnb
from accelerate import Accelerator

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

acc = Accelerator()
print("Accelerator device:", acc.device)


Torch: 2.5.1+cu121
CUDA available: True
GPU: Tesla T4
Accelerator device: cuda


In [ ]:
!accelerate config default



accelerate configuration saved at /root/.cache/huggingface/accelerate/default_config.yaml


In [3]:
# ======================
# CUDA FRAGMENTATION FIX (MUST BE FIRST)
# ======================
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# ======================
# IMPORTS
# ======================
import torch
import json
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    BitsAndBytesConfig,
    AutoConfig
)
from peft import LoraConfig, get_peft_model

# ======================
# CONFIG
# ======================
BASE_MODEL = "microsoft/phi-2"
OUTPUT_DIR = "./peft_pdf_extractor"

MAX_LENGTH = 512
BATCH_SIZE = 1
GRAD_ACCUM = 8
EPOCHS = 3
LR = 2e-4

PROMPT_TEMPLATE = """You are a healthcare document extraction system.
Extract the following 5 fields from the document.
If a value is missing, return null.
Return valid JSON only.

Document:
{document}

JSON:
"""

# ======================
# TOKENIZER
# ======================
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.model_max_length = MAX_LENGTH

# ======================
# QLORA CONFIG (4-BIT)
# ======================
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

# ======================
# MODEL
# ======================
config = AutoConfig.from_pretrained(BASE_MODEL, trust_remote_code=True)
config.pad_token_id = tokenizer.pad_token_id

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    config=config,
    quantization_config=bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True,
    trust_remote_code=True
)

# MEMORY SAVERS
model.gradient_checkpointing_enable()
model.config.use_cache = False

# ======================
# PEFT (LoRA)
# ======================
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj"]
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# ======================
# DATASET (DUMMY)
# ======================
with open("train.jsonl", "w") as f:
    f.write(json.dumps({
        "document": "Patient Name: John Doe, Age: 45, Diagnosis: Flu, Treatment: Tylenol",
        "output": {
            "patient_name": "John Doe",
            "age": "45",
            "diagnosis": "Flu",
            "treatment": "Tylenol",
            "doctor": None
        }
    }) + "\n")
    f.write(json.dumps({
        "document": "Name: Jane Smith, Diagnosis: Headache, Rx: Ibuprofen",
        "output": {
            "patient_name": "Jane Smith",
            "age": None,
            "diagnosis": "Headache",
            "treatment": "Ibuprofen",
            "doctor": None
        }
    }) + "\n")

dataset = load_dataset("json", data_files={"train": "train.jsonl"})

def preprocess(example):
    prompt = PROMPT_TEMPLATE.format(document=example["document"])
    completion = json.dumps(example["output"], ensure_ascii=False)

    text = prompt + completion
    tokens = tokenizer(
        text,
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length"
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

dataset = dataset.map(
    preprocess,
    remove_columns=dataset["train"].column_names
)

# ======================
# TRAINING
# ======================
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=EPOCHS,
    learning_rate=LR,
    bf16=True,
    fp16=False,
    optim="paged_adamw_8bit",
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=2,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    data_collator=DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=False
    )
)

trainer.train()

# ======================
# SAVE ADAPTER
# ======================
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]

trainable params: 2,621,440 || all params: 2,782,305,280 || trainable%: 0.0942


Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

Step,Training Loss


('./peft_pdf_extractor/tokenizer_config.json',
 './peft_pdf_extractor/tokenizer.json')

In [4]:
from peft import PeftModel

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="auto"
)

model = PeftModel.from_pretrained(base_model, OUTPUT_DIR)
model.eval()

pdf_text = "Policy No: ABC123\nCustomer: Rahul Sharma\nPremium: INR 12000"

prompt = PROMPT_TEMPLATE.format(document=pdf_text)
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0,
        do_sample=False
    )

print(tokenizer.decode(output[0], skip_special_tokens=True))


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


You are a healthcare document extraction system.
Extract the following 5 fields from the document.
If a value is missing, return null.
Return valid JSON only.

Document:
Policy No: ABC123
Customer: Rahul Sharma
Premium: INR 12000

JSON:
{
  "Policy No": "ABC123",
  "Customer": "Rahul Sharma",
  "Premium": "INR 12000"
}

Document:
Policy No: XYZ456
Customer:
Premium: INR 15000

JSON:
{
  "Policy No": "XYZ456",
  "Customer": null,
  "Premium": "INR 15000"
}

Document:
Policy No: PQR789
Customer: Rahul Sharma
Premium: INR 10000

JSON:
{
  "Policy No": "PQR789",
  "Customer": "Rahul Sharma",
  "Premium": "INR 10000"
}

Document:
Policy No: LMN012
Customer: Rahul Sharma
Premium: INR 12000

JSON:
{
  "Policy No": "LMN012",
  "Customer":


### 1. Install a single package

To install a single package, use the `!pip install` command followed by the package name.

In [ ]:
# Install the pandas library
!pip install pandas

### 2. Install multiple packages

You can install multiple packages at once by listing them separated by spaces.

In [ ]:
# Install numpy and matplotlib libraries
!pip install numpy matplotlib

### 3. Install a specific version of a package

To install a specific version, use `==` followed by the version number. This is useful for reproducibility or compatibility.

In [ ]:
# Install an older version of scikit-learn
!pip install scikit-learn==0.24.2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 46.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  error: subprocess-exited-with-error
  
  × Preparing metadata (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (pyproject.toml) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.


### 4. Upgrade an existing package

To upgrade an already installed package to its latest version, use the `-U` or `--upgrade` flag.

In [ ]:
# Upgrade pandas to its latest version
!pip install --upgrade pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 87.3 MB/s eta 0:00:00
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.0 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.0 which is incompatible.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, but you have pandas 3.0.0 which is incompatible.
db-dtypes 1.5.0 requires pandas<3.0.0,>=1.5.3, but you have pandas 3.0.0 which is incompatible.


### 5. Install from a `requirements.txt` file

If you have a `requirements.txt` file listing all your dependencies, you can install them all at once using the `-r` flag.

In [ ]:
# Create a dummy requirements.txt file for demonstration
%%writefile requirements.txt
pandas==2.2.0
scipy>=1.10.0

# Install packages from the requirements.txt file
!pip install -r requirements.txt

Writing requirements.txt


### Important Notes:

*   **`!` prefix**: In Colab (and Jupyter notebooks), the `!` prefix allows you to run shell commands directly.
*   **Restarting Runtime**: Sometimes, especially after installing certain complex packages or packages with native extensions, you might need to restart the Colab runtime (`Runtime > Restart runtime`) for the changes to take full effect.